# Regional run QA: integrity scan and small multiples

Quality control for the full South Africa regional run, across every GCM, scenario,
variable, and ensemble member the output stores contain. The notebook runs in four passes,
each answering a different question:

| Pass | Question it answers |
| --- | --- |
| Inventory | What leaves exist, how big are they, and over what time range? |
| Completeness | Did every leaf the QA configs ask for actually land? |
| Integrity scan | Does any leaf contain NaN, or values outside the plausible range? |
| Small multiples | Do the time series and maps look physically sensible? |

The integrity scan is the part that catches problems. Hundreds of panels cannot be
eyeballed reliably, so the scan reduces every leaf to a handful of numbers and reports a
verdict per leaf; the figures are there to interpret what the scan flags, not to find it.

## The halo

The run covers `[-38, -19, 13, 36]`, which is all of South Africa (`[-35, -22, 16, 33]`)
plus a roughly 3 degree halo. `subset_space` takes a hard slice with no padding, so regrid
and spatial disaggregation edge effects land on the outer boundary of the box. The halo
puts that boundary outside the region of interest.

Every number in this notebook is computed on the region of interest with the halo trimmed
off. The maps are the exception: they draw the full box and outline the region of interest,
so the halo stays visible as the discard band it is.

## Before running

Set `BRANCH` below to the branch you passed to `bcsd run --branch`. The stores are written
per branch, so a mismatch either raises or silently opens an older run.

## Setup

In [ ]:
import logging
import os
import pathlib

import dask
import frisky
import icechunk
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import zarr
from dask.distributed import Client
from dask_array.xarray import register
from IPython.display import Markdown, display

from srm.cache import ArtifactCache
from srm.cli import load_configs
from srm.config import SCENARIO_TO_GROUP
from srm.qaqc import VAR_SPATIAL_RANGES, area_weights

register()
zarr.config.set({"async.concurrency": 128})

os.environ["FRISKY_SUMMARY"] = "off"
os.environ["FRISKY_DEATH_DUMP_DIR"] = ""

logging.getLogger("distributed.worker.memory").setLevel(logging.ERROR)
pd.set_option("display.max_rows", 200)

In [ ]:
client = frisky.hijack(Client(n_workers=12))
client

## Configuration

`SUBSET_ID` is built with the same helper the pipeline uses to name the store, so the
notebook cannot drift from the configs. Bounds are floats because `BCSDConfig` coerces
`subset_bounds` to float, and the store name is built from the coerced value.

In [ ]:
# Must match the --branch passed to `bcsd run`.
BRANCH = "full-regional-run-issue-534"

# Must match subset_bounds in configs/qa/*/*.yaml.
SUBSET_BOUNDS = (-38.0, -19.0, 13.0, 36.0)

# All of South Africa. Everything quantitative is computed inside this box.
ROI_BOUNDS = (-35.0, -22.0, 16.0, 33.0)

BUCKET = "carbonplan-scratch"
OBS_DATASET = "ERA5"
GCMS = ("CESM2-WACCM", "MIROC-ES2H", "UKESM")


def repo_root(start: pathlib.Path | None = None) -> pathlib.Path:
    """Walk up to the directory holding pyproject.toml.

    Resolving from the working directory rather than a fixed relative path keeps the
    completeness pass working whether the kernel starts in docs/how-to or the repo root.
    """
    here = (start or pathlib.Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError(f"no pyproject.toml above {here}")


# Used by the completeness pass to learn which leaves to expect.
CONFIG_PATH = str(repo_root() / "configs" / "qa")

# Window used for the first/last climatology maps.
MAP_WINDOW_DAYS = 3650

SUBSET_ID = ArtifactCache._get_subset_id(SUBSET_BOUNDS)
PREFIXES = {gcm: f"srm/output/qa/{gcm}-{OBS_DATASET}-{SUBSET_ID}.icechunk" for gcm in GCMS}

print(f"branch    {BRANCH}")
print(f"subset_id {SUBSET_ID}")

## Open the stores

A store that does not exist yet, or a branch the run has not written, is reported rather
than raised. The run can still be in flight while this notebook is used to watch it land.

In [ ]:
def open_datatree(bucket: str, prefix: str, branch: str = BRANCH) -> xr.DataTree:
    storage = icechunk.s3_storage(bucket=bucket, prefix=prefix, from_env=True)
    repo = icechunk.Repository.open(storage)
    session = repo.readonly_session(branch=branch)
    return xr.open_datatree(session.store, engine="zarr", chunks="auto")


datatrees: dict[str, xr.DataTree] = {}
open_errors: dict[str, str] = {}

for name, prefix in PREFIXES.items():
    try:
        datatrees[name] = open_datatree(BUCKET, prefix)
    except Exception as exc:  # noqa: BLE001  # report every failure, do not stop at the first
        open_errors[name] = f"{type(exc).__name__}: {exc}"

for name, err in open_errors.items():
    print(f"FAILED {name}: {err}")
print(f"opened {len(datatrees)}/{len(PREFIXES)} stores on branch {BRANCH!r}")

## Pass 1: inventory

Walk every tree and record one row per leaf. `debiased_coarse` mirrors the scenario groups
one level down, so its paths are unpacked separately and tagged `stage="coarse"`; those are
the bias-corrected coarse intermediates written when `save_intermediate` is set, not the
downscaled product.

In [ ]:
def leaf_records(datatrees: dict[str, xr.DataTree]) -> list[dict]:
    """One record per leaf DataArray, carrying the lazy array for later reduction."""
    rows: list[dict] = []
    for model, dt in datatrees.items():
        for node in dt.subtree:
            if not node.is_leaf:
                continue
            ds_node = node.to_dataset()
            if not ds_node.data_vars:
                continue
            parts = node.path.strip("/").split("/")
            if parts[0] == "debiased_coarse":
                stage = "coarse"
                scenario = parts[1] if len(parts) > 1 else ""
                variable, member = (parts[2:4] + [None, None])[:2]
            else:
                stage = "fine"
                scenario = parts[0]
                variable, member = (parts[1:3] + [None, None])[:2]
            for da in ds_node.data_vars.values():
                time = da["time"]
                rows.append(
                    {
                        "gcm": model,
                        "stage": stage,
                        "scenario": scenario,
                        "variable": variable,
                        "member": member,
                        "ntime": int(time.size),
                        "start": str(time.values[0])[:10],
                        "end": str(time.values[-1])[:10],
                        "nlat": int(da.sizes.get("lat", 0)),
                        "nlon": int(da.sizes.get("lon", 0)),
                        "gib": da.nbytes / 2**30,
                        "units": da.attrs.get("units", ""),
                        "_da": da,
                    }
                )
    return rows


records = leaf_records(datatrees)
inventory = pd.DataFrame(records).drop(columns="_da")

print(f"{len(inventory)} leaves, {inventory.gib.sum():.1f} GiB uncompressed")
inventory.groupby(["gcm", "stage", "scenario"]).agg(
    leaves=("variable", "size"),
    variables=("variable", "nunique"),
    members=("member", "nunique"),
    first=("start", "min"),
    last=("end", "max"),
    gib=("gib", "sum"),
).round(1)

## Pass 2: completeness

Expand the QA configs into the set of leaves the run is supposed to produce, then diff it
against what the stores actually hold. This is the pass that distinguishes "the run is
still going" from "the run finished and quietly dropped something".

Mid-run, a non-empty `missing` list is normal. After the run reports success it should be
empty, and anything under `unexpected` means the store holds a leaf no current config asks
for, which usually means a stale write from an earlier run on the same branch.

In [ ]:
expected = {
    (cfg.gcm, SCENARIO_TO_GROUP[cfg.scenario], cfg.variable, cfg.ensemble_member)
    for cfg in load_configs(CONFIG_PATH)[0]
}
# Only compare GCMs whose store actually opened, so an unopened store is reported once
# above rather than as hundreds of missing leaves here.
expected = {key for key in expected if key[0] in datatrees}

actual = {
    (rec["gcm"], rec["scenario"], rec["variable"], rec["member"])
    for rec in records
    if rec["stage"] == "fine"
}

missing = sorted(expected - actual)
unexpected = sorted(actual - expected)

print(f"expected   {len(expected)} fine leaves")
print(f"found      {len(actual)}")
print(f"missing    {len(missing)}")
print(f"unexpected {len(unexpected)}")

cols = ["gcm", "scenario", "variable", "member"]
if missing:
    display(Markdown("**Missing leaves by group**"))
    display(pd.DataFrame(missing, columns=cols).groupby(["gcm", "scenario"]).size().to_frame("n"))
if unexpected:
    display(Markdown("**Unexpected leaves**"))
    display(pd.DataFrame(unexpected, columns=cols))

## Pass 3: integrity scan

Every reduction below is built lazily first and computed in a single `dask.compute`, so
each source chunk is read once and fed to every consumer that needs it. Computing per leaf
instead would re-read the same data for each statistic.

`srm.qa_checks.run_artifact_checks` implements the same checks the pipeline gates on, but
it takes a computed array and calls `np.isnan(da.values)`, which would pull every leaf into
client memory. The reductions here stay in dask and reuse `VAR_SPATIAL_RANGES`, the same
threshold table, so the verdicts match what the pipeline would say.

What each reduction is for:

| Reduction | Catches |
| --- | --- |
| `roi_nan_frac` | NaN inside the region of interest, the failure mode of issues #517 and #553 |
| `nan_frac` | NaN anywhere in the box, including the halo, where edge effects are expected |
| `vmin` / `vmax` | values outside the plausible range for the variable |
| `nan_by_time` | days that are partly but not wholly NaN, which is how a bad stitch shows up |
| `roi_annual` | the area-weighted annual series each figure below is drawn from |

In [ ]:
def clip_to_roi(da: xr.DataArray, bounds: tuple[float, ...] = ROI_BOUNDS) -> xr.DataArray:
    """Trim the halo, handling either latitude ordering."""
    lat_min, lat_max, lon_min, lon_max = bounds
    lat = da["lat"].values
    lat_slice = slice(lat_max, lat_min) if lat[0] > lat[-1] else slice(lat_min, lat_max)
    return da.sel(lat=lat_slice, lon=slice(lon_min, lon_max))


def leaf_reductions(da: xr.DataArray) -> dict[str, xr.DataArray]:
    roi = clip_to_roi(da)
    weights = area_weights(roi["lat"])
    window = min(MAP_WINDOW_DAYS, da.sizes["time"])
    return {
        "nan_frac": da.isnull().mean(),
        "roi_nan_frac": roi.isnull().mean(),
        "vmin": da.min(),
        "vmax": da.max(),
        "nan_by_time": roi.isnull().mean(("lat", "lon")),
        "roi_annual": roi.weighted(weights).mean(("lat", "lon")).resample(time="YS").mean(),
        # Maps keep the halo so the discard band stays visible.
        "map_first": da.isel(time=slice(0, window)).mean("time"),
        "map_last": da.isel(time=slice(-window, None)).mean("time"),
    }


REDUCTIONS = (
    "nan_frac",
    "roi_nan_frac",
    "vmin",
    "vmax",
    "nan_by_time",
    "roi_annual",
    "map_first",
    "map_last",
)

tasks = {
    (i, key): value
    for i, rec in enumerate(records)
    for key, value in leaf_reductions(rec["_da"]).items()
}
assert set(REDUCTIONS) == {key for _, key in tasks}, (
    "REDUCTIONS is out of step with leaf_reductions"
)
print(f"{len(tasks)} reductions over {inventory.gib.sum():.1f} GiB — this is the long cell")

In [ ]:
%%time
computed = dict(zip(tasks.keys(), dask.compute(*tasks.values())))
print(f"computed {len(computed)} reductions")

### Verdicts

One row per leaf, failures first. A leaf fails if it is entirely NaN, holds any NaN inside
the region of interest, has partially-NaN days, or falls outside the envelope
`VAR_SPATIAL_RANGES` defines for its variable.

NaN in the halo alone is reported in `nan_frac` but does not fail the leaf, because the
halo exists to absorb exactly that.

In [ ]:
def verdict(rec: dict, red: dict) -> dict:
    variable = rec["variable"]
    nan_frac = float(red["nan_frac"])
    roi_nan_frac = float(red["roi_nan_frac"])
    vmin, vmax = float(red["vmin"]), float(red["vmax"])

    nan_by_time = np.asarray(red["nan_by_time"])
    partial_nan_days = int(((nan_by_time > 0) & (nan_by_time < 1)).sum())

    issues: list[str] = []
    if nan_frac == 1.0:
        issues.append("all NaN")
    elif roi_nan_frac > 0:
        issues.append(f"{roi_nan_frac:.3%} NaN in ROI")
    if partial_nan_days:
        issues.append(f"{partial_nan_days} partial-NaN days")

    ranges = VAR_SPATIAL_RANGES.get(variable)
    if ranges is not None and not np.isnan(vmin):
        lo, hi = ranges["min"][0], ranges["max"][1]
        if vmin < lo or vmax > hi:
            issues.append(f"outside [{lo:g}, {hi:g}]")

    return {
        "gcm": rec["gcm"],
        "stage": rec["stage"],
        "scenario": rec["scenario"],
        "variable": variable,
        "member": rec["member"],
        "units": rec["units"],
        "nan_frac": nan_frac,
        "roi_nan_frac": roi_nan_frac,
        "partial_nan_days": partial_nan_days,
        "vmin": vmin,
        "vmax": vmax,
        "status": "FAIL" if issues else "PASS",
        "issues": "; ".join(issues),
    }


scan = pd.DataFrame(
    [verdict(rec, {key: computed[(i, key)] for key in REDUCTIONS}) for i, rec in enumerate(records)]
)
scan = scan.sort_values(
    ["status", "roi_nan_frac", "gcm", "scenario", "variable", "member"],
    ascending=[True, False, True, True, True, True],
)

n_fail = int((scan.status == "FAIL").sum())
print(f"{n_fail} FAIL / {len(scan)} leaves")
scan.groupby(["gcm", "stage", "status"]).size().to_frame("leaves")

### Failing leaves

In [ ]:
failures = scan[scan.status == "FAIL"]
if failures.empty:
    display(Markdown("**No leaf failed the integrity scan.**"))
else:
    display(
        failures[
            [
                "gcm",
                "stage",
                "scenario",
                "variable",
                "member",
                "roi_nan_frac",
                "partial_nan_days",
                "vmin",
                "vmax",
                "units",
                "issues",
            ]
        ].style.format({"roi_nan_frac": "{:.3%}", "vmin": "{:.4g}", "vmax": "{:.4g}"})
    )

### Observed ranges per variable

Values inside the envelope can still be wrong. This table shows what each variable actually
spans across the whole run so a plausible-but-suspicious range, such as `rsds` topping out
far below the expected clear-sky maximum, is visible even when nothing failed.

In [ ]:
ranges_seen = (
    scan[scan.stage == "fine"]
    .groupby(["variable", "units"])
    .agg(leaves=("vmin", "size"), observed_min=("vmin", "min"), observed_max=("vmax", "max"))
    .reset_index()
)
ranges_seen["envelope"] = ranges_seen.variable.map(
    lambda v: (
        f"[{VAR_SPATIAL_RANGES[v]['min'][0]:g}, {VAR_SPATIAL_RANGES[v]['max'][1]:g}]"
        if v in VAR_SPATIAL_RANGES
        else "no range defined"
    )
)
ranges_seen

## Pass 4: small multiples

### Palette and encoding

Scenarios carry identity, so they take the first three slots of the categorical palette in
fixed order. Those three validate on all pairs for both normal vision and the common
colorblind simulations, which matters here because the scenario lines overlap rather than
sitting side by side.

Ensemble members are not given their own hues. Ten hues cannot be told apart reliably, and
member identity is rarely the question. Each scenario is drawn as a band spanning the
ensemble range with the ensemble mean on top, so a member that leaves the family shows up
as a widening band rather than as a line the reader has to pick out of a legend.

In [ ]:
# Categorical slots 1-3, validated for all-pairs separation in light and dark mode.
SCENARIO_COLORS = {
    "ssp245": "#2a78d6",
    "g6_1p5k": "#eb6834",
    "g6_1p5k_end": "#1baf7a",
}
SCENARIO_FALLBACK = "#52514e"
VARIABLE_ORDER = ["tas", "tasmax", "tasmin", "dtr", "pr", "rsds", "hurs"]


def scenario_color(scenario: str) -> str:
    return SCENARIO_COLORS.get(scenario, SCENARIO_FALLBACK)


def order_variables(variables) -> list[str]:
    known = [v for v in VARIABLE_ORDER if v in variables]
    return known + sorted(set(variables) - set(known))


def recessive_axes(ax) -> None:
    ax.grid(alpha=0.25, lw=0.5)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.tick_params(labelsize=7)
    # Absolute values matter here. Matplotlib's default offset would label a tas axis
    # "+2.95e2" with ticks at 0.005, which is unreadable for a plausibility check; a
    # multiplicative scale factor is still allowed for small-magnitude variables like pr.
    ax.ticklabel_format(axis="y", useOffset=False)


def annual_by_scenario(gcm: str, variable: str, stage: str = "fine") -> dict[str, xr.DataArray]:
    """Stack each scenario's members onto a member dimension, aligned on year."""
    out: dict[str, xr.DataArray] = {}
    for i, rec in enumerate(records):
        if rec["gcm"] != gcm or rec["variable"] != variable or rec["stage"] != stage:
            continue
        out.setdefault(rec["scenario"], []).append(
            computed[(i, "roi_annual")].assign_coords(member=rec["member"])
        )
    return {
        scenario: xr.concat(das, dim="member", join="outer") for scenario, das in out.items() if das
    }

### Annual area-weighted means over the region of interest

One row per variable, all scenarios overlaid. Daily values at a single grid point were what
this notebook used to plot; an area-weighted annual mean over the region is both far less
noisy and far more likely to expose the failures this project actually hits, which are
discontinuities at scenario boundaries and members that disagree with their siblings.

Watch for a step at 2015 where the historical bridge hands over, a step at 2035 where the
SAI runs branch, and a step at 2085 where the termination run resumes.

In [ ]:
def plot_gcm_annual(gcm: str) -> plt.Figure:
    variables = order_variables(
        {rec["variable"] for rec in records if rec["gcm"] == gcm and rec["stage"] == "fine"}
    )
    fig, axes = plt.subplots(
        len(variables), 1, figsize=(9.5, 1.75 * len(variables)), sharex=True, squeeze=False
    )
    seen: dict[str, str] = {}

    for ax, variable in zip(axes[:, 0], variables):
        recessive_axes(ax)
        units = next(
            (r["units"] for r in records if r["gcm"] == gcm and r["variable"] == variable), ""
        )
        ax.set_ylabel(f"{variable}\n{units}", fontsize=8)

        for scenario, series in sorted(annual_by_scenario(gcm, variable).items()):
            color = scenario_color(scenario)
            seen[scenario] = color
            # Integer years rather than the time coord: the GCMs mix calendars (CESM is
            # noleap), and an integer axis renders identically for all of them.
            years = series["time"].dt.year.values
            lo = series.min("member").values
            hi = series.max("member").values
            mid = series.mean("member").values

            n_members = series.sizes.get("member", 1)
            if n_members > 1:
                ax.fill_between(years, lo, hi, color=color, alpha=0.18, lw=0)
            ax.plot(years, mid, color=color, lw=2, solid_capstyle="round")

    axes[-1, 0].set_xlabel("year", fontsize=8)
    handles = [
        plt.Line2D([], [], color=color, lw=2, label=scenario)
        for scenario, color in sorted(seen.items())
    ]
    if handles:
        fig.legend(
            handles=handles,
            loc="upper right",
            bbox_to_anchor=(0.995, 0.995),
            frameon=False,
            fontsize=8,
            ncols=len(handles),
        )
    fig.suptitle(
        f"{gcm} — annual area-weighted mean over {ROI_BOUNDS}, band spans ensemble range",
        fontsize=10,
        y=0.999,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    return fig


for gcm in sorted(datatrees):
    display(Markdown(f"#### {gcm}"))
    figure = plot_gcm_annual(gcm)
    display(figure)
    plt.close(figure)

### Climatology maps

For each scenario: the mean of the first and last ten years of the record, and the change
between them. The first two panels share a color scale per row so they can be compared
directly; the change panel is diverging and centered on zero.

The dashed rectangle is the region of interest. Everything outside it is halo, and structure
that appears only there is the edge effect the halo was added to absorb.

Single days were what this notebook used to map. A ten-year mean removes the weather so what
remains is the climate signal, which is what a scenario comparison is actually about.

In [ ]:
def add_roi_box(ax) -> None:
    lat_min, lat_max, lon_min, lon_max = ROI_BOUNDS
    ax.add_patch(
        plt.Rectangle(
            (lon_min, lat_min),
            lon_max - lon_min,
            lat_max - lat_min,
            fill=False,
            ec="#0b0b0b",
            lw=0.9,
            ls="--",
        )
    )


def show_map(ax, da: xr.DataArray, **kwargs):
    lat, lon = da["lat"].values, da["lon"].values
    image = ax.imshow(
        np.asarray(da),
        extent=[lon.min(), lon.max(), lat.min(), lat.max()],
        origin="lower" if lat[0] < lat[-1] else "upper",
        aspect="auto",
        **kwargs,
    )
    ax.set_xticks([])
    ax.set_yticks([])
    add_roi_box(ax)
    return image


def ensemble_mean_maps(gcm: str, scenario: str, variable: str) -> tuple:
    """Ensemble mean of the first and last climatology windows for one leaf group."""
    first, last = [], []
    for i, rec in enumerate(records):
        if (rec["gcm"], rec["scenario"], rec["variable"], rec["stage"]) != (
            gcm,
            scenario,
            variable,
            "fine",
        ):
            continue
        first.append(computed[(i, "map_first")])
        last.append(computed[(i, "map_last")])
    if not first:
        return None, None

    def stack(das: list[xr.DataArray]) -> xr.DataArray:
        return xr.concat(das, dim="member", join="outer").mean("member")

    return stack(first), stack(last)


def plot_scenario_maps(gcm: str, scenario: str) -> plt.Figure | None:
    variables = order_variables(
        {
            rec["variable"]
            for rec in records
            if rec["gcm"] == gcm and rec["scenario"] == scenario and rec["stage"] == "fine"
        }
    )
    if not variables:
        return None

    fig, axes = plt.subplots(
        len(variables), 3, figsize=(8.5, 2.3 * len(variables)), squeeze=False, layout="constrained"
    )
    for row, (variable, axrow) in enumerate(zip(variables, axes)):
        first, last = ensemble_mean_maps(gcm, scenario, variable)
        if first is None:
            for ax in axrow:
                ax.set_xticks([])
                ax.set_yticks([])
            continue

        units = next(
            (r["units"] for r in records if r["gcm"] == gcm and r["variable"] == variable), ""
        )
        pooled = np.concatenate([np.asarray(first).ravel(), np.asarray(last).ravel()])
        vmin, vmax = np.nanpercentile(pooled, [2, 98])

        # Sequential magnitude: viridis is single-direction in lightness and colorblind
        # safe, unlike the rainbow maps this replaces.
        for ax, da, label in ((axrow[0], first, "first decade"), (axrow[1], last, "last decade")):
            image = show_map(ax, da, vmin=vmin, vmax=vmax, cmap="viridis")
            if row == 0:
                ax.set_title(label, fontsize=8)
        cbar = fig.colorbar(image, ax=axrow[:2].tolist(), fraction=0.03, pad=0.01)
        cbar.ax.tick_params(labelsize=6)
        cbar.set_label(units, fontsize=7)

        # Diverging change, centered on zero with a neutral midpoint. A record shorter
        # than twice the window makes the two windows overlap, so the change panel would
        # be misleadingly flat; say so rather than drawing a blank map.
        change = last - first
        ntime = min(
            r["ntime"]
            for r in records
            if (r["gcm"], r["scenario"], r["variable"], r["stage"])
            == (gcm, scenario, variable, "fine")
        )
        if ntime < 2 * MAP_WINDOW_DAYS:
            axrow[2].set_xlabel("windows overlap", fontsize=6, color="#52514e")
        limit = float(np.nanpercentile(np.abs(np.asarray(change)), 98)) or 1.0
        image = show_map(axrow[2], change, vmin=-limit, vmax=limit, cmap="RdBu_r")
        if row == 0:
            axrow[2].set_title("change", fontsize=8)
        cbar = fig.colorbar(image, ax=axrow[2], fraction=0.05, pad=0.01)
        cbar.ax.tick_params(labelsize=6)
        cbar.set_label(units, fontsize=7)

        axrow[0].set_ylabel(variable, fontsize=8)

    fig.suptitle(
        f"{gcm} / {scenario} — {MAP_WINDOW_DAYS // 365}-year climatology, "
        "dashed box is the region of interest",
        fontsize=10,
    )
    return fig


for gcm in sorted(datatrees):
    display(Markdown(f"#### {gcm}"))
    scenarios = sorted(
        {rec["scenario"] for rec in records if rec["gcm"] == gcm and rec["stage"] == "fine"}
    )
    for scenario in scenarios:
        display(Markdown(f"##### {scenario}"))
        figure = plot_scenario_maps(gcm, scenario)
        if figure is None:
            continue
        display(figure)
        plt.close(figure)

### Per-member spread on the last decade

The band in the annual figure shows how wide the ensemble is but not which member sits at
the edge. This grid maps the last-decade climatology per member on a shared color scale per
row, so a member that departs from its siblings is visible as a panel that does not match.

In [ ]:
def plot_member_maps(gcm: str, scenario: str) -> plt.Figure | None:
    subset = [
        (i, rec)
        for i, rec in enumerate(records)
        if rec["gcm"] == gcm and rec["scenario"] == scenario and rec["stage"] == "fine"
    ]
    if not subset:
        return None
    variables = order_variables({rec["variable"] for _, rec in subset})
    members = sorted({rec["member"] for _, rec in subset})
    index = {(rec["variable"], rec["member"]): i for i, rec in subset}

    fig, axes = plt.subplots(
        len(variables),
        len(members),
        figsize=(1.5 * len(members) + 1.2, 1.9 * len(variables)),
        squeeze=False,
        layout="constrained",
    )
    for variable, axrow in zip(variables, axes):
        das = [
            computed[(index[(variable, m)], "map_last")] for m in members if (variable, m) in index
        ]
        if das:
            pooled = np.concatenate([np.asarray(d).ravel() for d in das])
            vmin, vmax = np.nanpercentile(pooled, [2, 98])
        else:
            vmin = vmax = None

        image = None
        for ax, member in zip(axrow, members):
            key = (variable, member)
            if key not in index:
                ax.set_xticks([])
                ax.set_yticks([])
                continue
            image = show_map(
                ax, computed[(index[key], "map_last")], vmin=vmin, vmax=vmax, cmap="viridis"
            )
            if variable == variables[0]:
                ax.set_title(member, fontsize=7)
        axrow[0].set_ylabel(variable, fontsize=8)
        if image is not None:
            cbar = fig.colorbar(image, ax=axrow.tolist(), fraction=0.02, pad=0.01)
            cbar.ax.tick_params(labelsize=6)

    fig.suptitle(f"{gcm} / {scenario} — last-decade climatology per member", fontsize=10)
    return fig


for gcm in sorted(datatrees):
    display(Markdown(f"#### {gcm}"))
    scenarios = sorted(
        {rec["scenario"] for rec in records if rec["gcm"] == gcm and rec["stage"] == "fine"}
    )
    for scenario in scenarios:
        display(Markdown(f"##### {scenario}"))
        figure = plot_member_maps(gcm, scenario)
        if figure is None:
            continue
        display(figure)
        plt.close(figure)

## Summary

In [ ]:
print(f"branch      {BRANCH}")
print(f"subset_id   {SUBSET_ID}")
print(f"stores      {len(datatrees)}/{len(PREFIXES)} opened")
print(f"leaves      {len(scan)} ({int((scan.stage == 'fine').sum())} fine)")
print(f"completeness {len(missing)} missing, {len(unexpected)} unexpected")
print(
    f"integrity   {int((scan.status == 'FAIL').sum())} FAIL, {int((scan.status == 'PASS').sum())} PASS"
)

if not failures.empty:
    display(Markdown("**Failing leaves by issue**"))
    display(failures.groupby("issues").size().sort_values(ascending=False).to_frame("leaves"))